# Week 2: Anatomical Curve Matching

This notebook explains how curves were matched across observers. It does not assume that the first angle in one observer's list corresponds to the first angle in another observer's list.

**Primary rule:** maximize one-to-one vertebral-interval overlap within each image and retain pairs with intersection-over-union (IoU) of at least 0.50.

## 1. Why matching is necessary

One radiograph can contain different curve counts across observers. Comparing `angles[0]` across everyone could therefore pair anatomically different curves. Each curve is represented by its upper and lower encoded vertebral endpoint.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Works when Jupyter starts from either the repository root or notebooks/.
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.resolve()))
TABLES = PROJECT_ROOT / "outputs" / "tables"

## 2. Interval similarity

For intervals \(A\) and \(B\):

\[
\mathrm{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}.
\]

IoU equals 1 for identical intervals and 0 when they do not overlap. The Hungarian algorithm then finds the set of one-to-one assignments with the greatest total IoU.

In [ ]:
from src.curve_matching import interval_iou

# Similar upper curves: [1, 6] and [1, 5]
interval_iou(1, 6, 1, 5)

In [ ]:
sensitivity = pd.read_csv(TABLES / "matching_sensitivity.csv")
sensitivity

## 3. Verified results

At IoU 0.50, the algorithm retained:

- **5,554** human spline curve pairs, with **86.1%** symmetric coverage;
- **5,189** human manual curve pairs, with **86.4%** coverage; and
- **3,684** neural-network-to-human spline pairs, with **87.2%** coverage.

![Curve-match coverage across thresholds](../outputs/figures/matching_sensitivity.png)

Coverage remains close to 89% at IoU 0.30 and approximately 86%–87% at 0.50, then declines at the stricter 0.70 threshold. This sensitivity analysis makes the threshold choice visible rather than hiding it.

## 4. Quality checks

- Every accepted pair meets the 0.50 threshold.
- A curve is never reused within the same observer-pair and image comparison.
- Match counts decrease when the threshold becomes stricter.
- Manual, human-spline, and neural-human comparisons are matched separately.

The rule is transparent and reproducible, but IoU is still an analytical proxy for anatomical correspondence rather than a clinical gold standard.